In [2]:
from pathlib import Path
from io import StringIO
import re
import pandas as pd
from bs4 import BeautifulSoup

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)
women_source = Path('../data/raw/championships/bigwest_2026_women.html')
men_source = Path('../data/raw/championships/bigwest_2026_men.html')


In [3]:
FIELD_KEYWORDS = ('long jump', 'triple jump', 'high jump', 'pole vault',
                  'shot put', 'discus', 'hammer', 'javelin')
COMBINED_KEYWORDS = ('heptathlon', 'decathlon', 'pentathlon')

def get_event_info(table_tag):
    heading = table_tag.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    while heading:
        text = heading.get_text(' ', strip=True)
        if re.search(r"(?:Women['’]s|Men['’]s)", text, re.I):
            break
        heading = heading.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    if not heading:
        return None, None, None, None
    text = heading.get_text(' ', strip=True)
    event = re.split(r'\b(?:Finals?|Preliminaries|Semi-Finals?|Flight|Heat|Section)\b',
                     text, maxsplit=1, flags=re.I)[0]
    event = re.sub(r"^(?:Women['’]s|Men['’]s)\s+", '', event).strip()
    low = event.lower()
    event_type = ('combined' if any(x in low for x in COMBINED_KEYWORDS)
                  else 'field' if any(x in low for x in FIELD_KEYWORDS)
                  else 'running')
    round_name = ('Flight' if re.search(r'\bFlight\b', text, re.I)
                  else 'Heat' if re.search(r'\bHeat\b', text, re.I)
                  else 'Preliminaries' if re.search(r'\bPreliminaries\b', text, re.I)
                  else 'Finals' if re.search(r'\bFinals?\b', text, re.I)
                  else 'Other')
    return event, event_type, round_name, text

# Main overall results tables for the 2026 women's saved HTML.
# Indices are specific to this file; check them when changing years/genders.
WOMEN_FIELD_TABLES = {
    'High Jump': 48, 'Pole Vault': 49, 'Long Jump': 50,
    'Triple Jump': 53, 'Shot Put': 56, 'Discus': 59,
    'Hammer': 62, 'Javelin': 65,
}

# Only Triple Jump MARK.2 has been independently checked against the
# previously verified results. Other multi-MARK selections need review.
WOMEN_VERIFIED_MARK_COLUMNS = {'Triple Jump': 'MARK.2', 'Discus': 'MARK', 'Hammer': 'MARK'}

def is_field_summary_table(df):
    return {'PL', 'NAME', 'TEAM', 'MARK'}.issubset(df.columns)

def get_official_mark_column(df, event_type, round_name, event=None, verified_marks=None):
    if event_type == 'field':
        cols = [c for c in df.columns if re.fullmatch(r'MARK(?:\.\d+)?', str(c))]
        if not cols:
            return None
        return (verified_marks or {}).get(event, cols[-1])
    cols = [c for c in df.columns if re.fullmatch(r'TIME(?:\.\d+)?', str(c))]
    if not cols:
        return None
    # Running-time selection is provisional and should be independently checked.
    return cols[0] if round_name in ('Flight', 'Heat', 'Preliminaries') else cols[-1]

# Men's main field tables are discovered by event heading, not by women's table numbers.
def find_main_field_tables(tables, table_tags):
    found = {}
    for i, (table, tag) in enumerate(zip(tables, table_tags)):
        event, event_type, round_name, _ = get_event_info(tag)
        if (event_type == 'field' and round_name not in ('Flight', 'Heat', 'Preliminaries')
                and is_field_summary_table(table)):
            if event in found:
                raise ValueError(f'Multiple main field tables for {event}: {found[event]}, {i}')
            found[event] = i
    return found


In [4]:
# Women's main field table numbers are known from the previously inspected HTML.
print('Women main field tables:', WOMEN_FIELD_TABLES)
print('Men main field tables will be detected from the saved men HTML.')


Women main field tables: {'High Jump': 48, 'Pole Vault': 49, 'Long Jump': 50, 'Triple Jump': 53, 'Shot Put': 56, 'Discus': 59, 'Hammer': 62, 'Javelin': 65}
Men main field tables will be detected from the saved men HTML.


In [5]:
def clean_event_table(table, tag, index, gender, field_tables, verified_marks):
    event, event_type, round_name, heading = get_event_info(tag)
    if not event or event_type == 'combined':
        return None
    df = table.copy()
    if event_type == 'field':
        # Use only the main overall results table, not the flight tables.
        if index != field_tables.get(event):
            return None
        if not is_field_summary_table(df):
            return None
    elif not {'PL', 'NAME', 'TEAM'}.issubset(df.columns):
        return None

    mark_col = get_official_mark_column(df, event_type, round_name, event, verified_marks)
    if mark_col is None:
        return None
    df = df.rename(columns={'PL':'place', 'NAME':'athlete_name', 'YEAR':'class_year',
                            'TEAM':'school', 'SC':'points', 'WIND':'wind',
                            'CONV':'converted_mark', mark_col:'mark'})
    ids = []
    for tr in tag.select('tbody tr'):
        link = tr.select_one('a[href*="/athletes/"]')
        m = re.search(r'/athletes/(\d+)', link.get('href', '')) if link else None
        ids.append(m.group(1) if m else None)
    if len(ids) != len(df):
        print(f'SKIP table {index}: athlete-link rows {len(ids)} != parsed rows {len(df)}')
        return None
    df['athlete_id'] = ids
    df = df[df['athlete_id'].notna() & df['athlete_name'].notna() & df['school'].notna()].copy()
    if df.empty:
        return None
    for optional in ('class_year', 'wind', 'converted_mark'):
        if optional not in df: df[optional] = None
    if 'points' not in df: df['points'] = 0
    df['points'] = pd.to_numeric(df['points'], errors='coerce').fillna(0)
    df['place'] = pd.to_numeric(df['place'], errors='coerce')
    df['year'], df['gender'], df['event'] = 2026, gender, event
    df['event_type'], df['round'] = event_type, round_name
    df['source_table'], df['source_heading'] = index, heading
    df['mark_source_column'] = mark_col
    return df[['year','gender','event','event_type','round','athlete_id',
               'athlete_name','class_year','school','place','mark','converted_mark',
               'points','wind','source_table','source_heading','mark_source_column']]


def process_championship(path, gender, field_tables=None, verified_marks=None):
    if not path.exists():
        raise FileNotFoundError(f'Missing saved {gender} HTML: {path}')
    soup = BeautifulSoup(path.read_text(encoding='utf-8'), 'html.parser')
    # Validate actual results headings, not the file name.
    event_headings = [h.get_text(' ', strip=True) for h in soup.find_all(['h1','h2','h3','h4','h5'])]
    expected = r"Women['’]s" if gender == 'Women' else r"Men['’]s"
    other = r"Men['’]s" if gender == 'Women' else r"Women['’]s"
    if not any(re.search(expected, h, re.I) for h in event_headings):
        raise ValueError(f'{path} does not contain {gender} event headings. Save the correct {gender} results page.')
    if any(re.search(other, h, re.I) for h in event_headings):
        print(f'Warning: {path} also contains opposite-gender headings; verify saved page.')
    for tag in soup.find_all(attrs={'colspan': True}):
        if not str(tag['colspan']).isdigit():
            tag['colspan'] = '1'
    table_tags = soup.find_all('table')
    tables = pd.read_html(StringIO(str(soup)))
    if len(table_tags) != len(tables):
        raise ValueError('HTML tables and pandas tables are not aligned')
    if field_tables is None:
        field_tables = find_main_field_tables(tables, table_tags)
    print(f'{gender} field tables:', field_tables)
    expected_fields = set(WOMEN_FIELD_TABLES)
    missing_fields = expected_fields - set(field_tables)
    if missing_fields:
        print(f'WARNING: {gender} missing main field tables:', sorted(missing_fields))
    parsed = []
    for i, (table, tag) in enumerate(zip(tables, table_tags)):
        result = clean_event_table(table, tag, i, gender, field_tables, verified_marks or {})
        if result is not None:
            parsed.append(result)
    if not parsed:
        raise ValueError(f'No athlete results extracted from {path}')
    results = pd.concat(parsed, ignore_index=True)
    priority = {'Finals': 4, 'Other': 3, 'Preliminaries': 2, 'Flight': 1, 'Heat': 1}
    results['round_priority'] = results['round'].map(priority).fillna(0)
    results['has_mark'] = results['mark'].notna() & results['mark'].astype(str).str.strip().ne('')
    results = results.sort_values(['round_priority', 'has_mark', 'points', 'source_table'],
                                  ascending=[False, False, False, True])
    results = results.drop_duplicates(subset=['year','gender','event','athlete_id'], keep='first')
    field = results[results['event_type'] == 'field'].copy()
    running = results[results['event_type'] == 'running'].copy()
    assert field.apply(lambda r: int(r['source_table']) == field_tables.get(r['event']), axis=1).all()
    return running, field

women_running, women_field = process_championship(
    women_source, 'Women', WOMEN_FIELD_TABLES, WOMEN_VERIFIED_MARK_COLUMNS)
men_running, men_field = process_championship(
    men_source, 'Men')  # Men mark columns are provisional and require verification.

running_df = pd.concat([women_running, men_running], ignore_index=True)
field_df = pd.concat([women_field, men_field], ignore_index=True)
print('Combined running rows:', len(running_df))
print('Combined field rows:', len(field_df))


Women field tables: {'High Jump': 48, 'Pole Vault': 49, 'Long Jump': 50, 'Triple Jump': 53, 'Shot Put': 56, 'Discus': 59, 'Hammer': 62, 'Javelin': 65}
Men field tables: {'High Jump': 41, 'Pole Vault': 42, 'Long Jump': 43, 'Triple Jump': 46, 'Shot Put': 49, 'Discus': 52, 'Hammer': 55, 'Javelin': 58}
Combined running rows: 483
Combined field rows: 307


In [6]:
# Both genders are in the same running and field dataframes.
assert set(running_df['gender']) == {'Women', 'Men'}
assert set(field_df['gender']) == {'Women', 'Men'}
print('RUNNING BY GENDER AND EVENT')
display(running_df.groupby(['gender', 'event']).size().rename('athletes'))
print('FIELD BY GENDER AND EVENT')
display(field_df.groupby(['gender', 'event']).agg(
    athletes=('athlete_id', 'nunique'), points=('points', 'sum'),
    source_table=('source_table', 'first'), mark_column=('mark_source_column', 'first')))


RUNNING BY GENDER AND EVENT


gender  event            
Men     10,000 Meters        20
        100 Meters           28
        110 Hurdles          11
        1500 Meters          18
        200 Meters           22
        3000 Steeplechase    18
        400 Hurdles          16
        400 Meters           23
        5000 Meters          37
        800 Meters           21
Women   10,000 Meters        23
        100 Hurdles          19
        100 Meters           36
        1500 Meters          23
        200 Meters           34
        3000 Steeplechase    18
        400 Hurdles          22
        400 Meters           29
        5000 Meters          38
        800 Meters           27
Name: athletes, dtype: int64

FIELD BY GENDER AND EVENT


athletes  points  source_table mark_column
gender event                                                  
Men    Discus             20    39.0            52      MARK.2
       Hammer             21    39.0            55      MARK.2
       High Jump          13    39.0            41      MARK.2
       Javelin            14    39.0            58      MARK.5
       Long Jump          25    39.0            43      MARK.2
       Pole Vault         12    39.0            42      MARK.5
       Shot Put           22    39.0            49      MARK.3
       Triple Jump        16    39.0            46      MARK.1
Women  Discus             24    39.0            59        MARK
       Hammer             24    39.0            62        MARK
       High Jump          18    39.0            48      MARK.3
       Javelin            17    39.0            65      MARK.4
       Long Jump          25    39.0            50      MARK.3
       Pole Vault         15    39.0            49      MARK.4
       Shot Put           23    39.0            56      MARK.6
       Triple Jump        18    39.0            53      MARK.2

In [7]:
print('FIELD MARK AUDIT — verify the official mark column and values before modeling.')
for gender in ('Women', 'Men'):
    for event in ('High Jump', 'Pole Vault', 'Long Jump', 'Triple Jump', 'Shot Put', 'Discus', 'Hammer', 'Javelin'):
        sub = field_df[(field_df['gender'] == gender) & (field_df['event'] == event)]
        if sub.empty:
            print('MISSING:', gender, event)
        else:
            print(gender, event, '| mark column:', sub['mark_source_column'].iloc[0])
            display(sub[['athlete_name','place','mark','points']].sort_values('place').head(3))
print('NOTE: Unverified field MARK selections and running TIME selections are provisional.')


FIELD MARK AUDIT — verify the official mark column and values before modeling.
Women High Jump | mark column: MARK.3


,athlete_name,place,mark,points
0,Zharia Taylor,1.0,1.78m,10.0
8,Lilian Turban,2.0,1.78m,8.0
16,Charli Kennedy,3.0,1.75m,6.0


Women Pole Vault | mark column: MARK.4


,athlete_name,place,mark,points
1,Lexi Evans,1.0,3.50m,10.0
9,Jonelle Scott,2.0,3.22m,8.0
17,Atianna Marquez,3.0,2.98m,6.0


Women Long Jump | mark column: MARK.3


...

Men Hammer | mark column: MARK.2


,athlete_name,place,mark,points
170,Travis Martin,1.0,82.49m,10.0
178,Javier Toro,2.0,61.06m,8.0
186,Tylor Connelly,3.0,50.26m,6.0


Men Javelin | mark column: MARK.5


,athlete_name,place,mark,points
171,Conrad Bernstein,1.0,71.25m,10.0
179,Matthew Bender,2.0,47.97m,8.0
187,Kyle Wood,3.0,47.67m,6.0


NOTE: Unverified field MARK selections and running TIME selections are provisional.


In [8]:
# Export two combined datasets, not separate files per gender.
running_path = output_dir / 'bigwest_2026_running.csv'
field_path = output_dir / 'bigwest_2026_field.csv'
running_df.to_csv(running_path, index=False)
field_df.to_csv(field_path, index=False)
print('Saved:', running_path, field_path, sep='\n')


Saved:
..\data\processed\bigwest_2026_running.csv
..\data\processed\bigwest_2026_field.csv


In [9]:
field_df

,year,gender,event,event_type,round,athlete_id,athlete_name,class_year,school,place,mark,converted_mark,points,wind,source_table,source_heading,mark_source_column,round_priority,has_mark
0,2026,Women,High Jump,field,Other,8999688,Zharia Taylor,SR-4,UC Irvine,1.0,1.78m,"5' 10""",10.0,None,48,Women's High Jump,MARK.3,3,True
1,2026,Women,Pole Vault,field,Other,8355014,Lexi Evans,SR-4,Cal Poly,1.0,3.50m,"14' 4""",10.0,None,49,Women's Pole Vault,MARK.4,3,True
2,2026,Women,Long Jump,field,Other,8364380,Kylee Davis,JR-3,UC Davis,1.0,7.45m,"20' 6.5""",10.0,+0.0,50,Women's Long Jump,MARK.3,3,True
3,2026,Women,Triple Jump,field,Other,8351002,Chelsea Aninyei,JR-3,UC Riverside,1.0,13.04m,"42' 9.5""",10.0,0.3,53,Women's Triple Jump,MARK.2,3,True
4,2026,Women,Shot Put,field,Other,8358314,Nyla Baker,SR-4,UC Irvine,1.0,18.90m,"51' 8.25""",10.0,None,56,Women's Shot Put,MARK.6,3,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,2026,Men,Javelin,field,Other,9042399,Hudson Shaver,SO-2,UC Santa Barbara,10.0,60.22m,"184' 8""",0.0,None,58,Men's Javelin,MARK.5,3,True
303,2026,Men,Javelin,field,Other,8998460,Ryan Gregory,SR-4,Long Beach St.,11.0,61.63m,"183' 10""",0.0,None,58,Men's Javelin,MARK.5,3,True
304,2026,Men,Javelin,field,Other,8361653,Milo Orona,SR-4,Long Beach St.,12.0,70.28m,"181' 7""",0.0,None,58,Men's Javelin,MARK.5,3,True
305,2026,Men,Javelin,field,Other,9277435,Sam Betz,JR-3,UC San Diego,13.0,63.32m,"173' 1""",0.0,None,58,Men's Javelin,MARK.5,3,True


: 